Demo — Reinforcement Learning para Trading
¶
Aula 7 | AI Engineering | MBA


Vamos formular Algorithmic Trading como 
MDP
 e treinar um agente 
Q-Learning tabular
 sobre dados sintéticos de FICT3.




⚠ Objetivo pedagógico: entender a 
formulação
 do problema. Q-Learning tabular é didático mas insuficiente para mercados reais.

1. Setup e dados
¶

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Fixar a semente aleatória é fundamental em Reinforcement Learning (RL) 
# para garantir que os resultados sejam reproduzíveis em sala de aula.
SEED = 7
np.random.seed(SEED)

# Lendo o arquivo CSV do ativo fictício "FICT3"
# O parâmetro parse_dates converte a coluna 'date' automaticamente para o tipo datetime.
df = pd.read_csv("ohlcv_ativo.csv", parse_dates=["date"]).sort_values("date").reset_index(drop=True)

# Calculando o retorno diário.
# pct_change() calcula a variação percentual em relação à linha anterior (fechamento de hoje / fechamento de ontem - 1).
# fillna(0) lida com o primeiro dia (que não tem dia anterior), atribuindo 0% de retorno.
df["ret"] = df["close"].pct_change().fillna(0)

# Imprimindo as características do dataset (período analisado e a quantidade de dias).
print(f"Periodo: {df['date'].min().date()} -> {df['date'].max().date()} | {len(df)} dias")

# Inspecionando as primeiras linhas.
df.head()

Esse conjunto de dados é de 750 pregoes de um ativo fictício chamado FICT3 gerado via Movimento Browniano Geométrico. 

O FICT3 é puro ruído gaussiano. Nao tem padrao real, reversao à média e nem momentum verdadeiro. O objetivo é puramente didático. Na prática, se o agente encontrar padroes, é overfit. 

estamos analisando o retorno por conta do MDP (requer estacionariedade - média e variancia estáveis no tempo). 

In [ ]:
# Criando uma figura com dois eixos (subplots) empilhados. sharex=True garante 
# que eles compartilhem o mesmo eixo X (datas), facilitando a comparação.
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)

# Primeiro subplot (axes[0]): Preço de fechamento da ação FICT3
axes[0].plot(df["date"], df["close"], color="#0A2540")
axes[0].set_title("Preco de fechamento - FICT3")
axes[0].set_ylabel("R$")

# Segundo subplot (axes[1]): Volume diário de ações negociadas.
# Usamos um gráfico de barras (bar) em vez de linha para visualizar o volume tradicionalmente.
axes[1].bar(df["date"], df["volume"], color="#000000", width=0.2)
axes[1].set_title("Volume diario")
axes[1].set_ylabel("acoes")

plt.tight_layout() # Ajusta as margens para não haver sobreposição.
plt.show()

2. Ambiente: Estado, Ação, Recompensa
¶


Estado
 $s$: quintil da média móvel 5d dos retornos (5 estados discretos)


Ação
 $a$: 0 = flat, 1 = long, 2 = short


Recompensa
 $r$: retorno do dia seguinte com sinal conforme a ação


Horizonte
: 1 dia (decisão diária)

In [ ]:
# Calculamos o momento (tendência) dos últimos 5 dias por meio de uma Média Móvel 
# (rolling) sobre a coluna de retornos diários.
df["mom5"] = df["ret"].rolling(5).mean() 

# O Q-Learning clássico não lida bem com estados contínuos. Precisamos de estados "discretos".
# pd.qcut divide o momentum de 5 dias em 5 "cestas" (quintis) com tamanho estatisticamente similar.
# labels=False força a saída a ser valores inteiros: 0, 1, 2, 3 e 4.
df["state"] = pd.qcut(df["mom5"], q=5, labels=False)

# O cálculo do rolling gera NaNs nas 4 primeiras linhas. Precisamos removê-las para o RL não quebrar.
df = df.dropna().reset_index(drop=True)
df["state"] = df["state"].astype(int) # Garantindo que o estado é um número inteiro

# Plotando a distribuição dos estados para confirmar se o qcut balanceou bem as observações.
fig, ax = plt.subplots(figsize=(7, 3.5))
df["state"].value_counts().sort_index().plot(kind="bar", ax=ax, color="#0A2540")
ax.set_title("Distribuicao dos estados (quintis de momentum 5d)")
ax.set_xlabel("Estado")
ax.set_ylabel("# observacoes")
plt.tight_layout()
plt.show()

# Resumo dos estados (como o modelo enxergará o mercado):
# 0 - forte queda
# 1 - queda leve
# 2 - lateral (sem forte direção)
# 3 - alta leve
# 4 - forte alta

3. Q-Learning Tabular
¶

In [ ]:
# Variáveis e hiperparâmetros do processo de Decisão de Markov (MDP)
n_states  = 5
n_actions = 3   # O agente terá 3 botões: 0 = Ficar de fora (flat), 1 = Comprar (long), 2 = Vender a descoberto (short)
alpha     = 0.10 # Learning Rate (Taxa de aprendizado): quanto da nova informação substitui a antiga
gamma     = 0.95 # Discount Factor: importância dada a recompensas futuras vs imediatas (próximo a 1 = longo prazo)

# Parâmetros de exploração (Epsilon-Greedy). 
# O agente começa explorando ações aleatórias (30%) e aos poucos se baseia apenas no que aprendeu (1%).
eps_start, eps_end = 0.30, 0.01 
n_episodes = 300 # Um episódio = percorrer a base de dados de treinamento inteira.

# Dividindo a base entre Treino (70%) e Teste (Out-of-sample) (30%)
split = int(0.7 * len(df))
train = df.iloc[:split].reset_index(drop=True)
test  = df.iloc[split:].reset_index(drop=True)

# Função de recompensa (Reward). A forma como o agente é pontuado pelas suas ações.
def reward(action, ret_next):
    if action == 1: return ret_next    # Se comprou (long), a recompensa é o retorno do dia seguinte.
    if action == 2: return -ret_next   # Se vendeu (short), lucra se o ativo cair (recompensa é o retorno invertido).
    return 0.0                         # Se ficou flat (fora), não ganha e não perde.

# Inicializando a "Mente" do agente: A Q-Table. Uma matriz preenchida com Zeros.
# Teremos 5 linhas (estados) x 3 colunas (ações).
Q = np.zeros((n_states, n_actions))
returns_per_episode = []

# Loop principal: Treinando através dos episódios
for ep in range(n_episodes):
    # Decaimento linear do epsilon. A cada episódio o agente explora um pouco menos aleatoriamente.
    eps = eps_end + (eps_start - eps_end) * (1 - ep / n_episodes)
    ep_reward = 0.0
    
    # Percorrendo cada dia da base de treinamento
    for i in range(len(train) - 1):
        # iat é usado para buscar um valor único de forma extremamente rápida.
        s      = int(train["state"].iat[i])      # Estado de hoje
        s_next = int(train["state"].iat[i+1])    # Estado de amanhã (necessário para o termo gamma * max(Q))
        r_next = float(train["ret"].iat[i+1])    # O retorno real que ocorreu no dia seguinte

        # Epsilon-Greedy: Escolha da Ação
        if np.random.rand() < eps:
            a = np.random.randint(n_actions)     # Exploração: Escolhe ação 0, 1 ou 2 de forma aleatória
        else:
            a = int(np.argmax(Q[s]))             # Explotação: Escolhe a ação que tem o maior valor esperado (maior Q) para o estado atual

        # Passo do Agente (Step)
        r = reward(a, r_next)                    # Calcula o quanto ganhou/perdeu
        
        # Atualização da Q-Table
        # Novo_Q = Q_Antigo + alpha * (recompensa_imediata + (gamma * maior_valor_futuro) - Q_Antigo)
        Q[s, a] += alpha * (r + gamma * Q[s_next].max() - Q[s, a])
        
        ep_reward += r
        
    returns_per_episode.append(ep_reward)

print("Q-table final:")
print(np.round(Q, 4))

Detalhando a equacao de Bellman:

* $\alpha$: taxa de aprendizado - o quanto a nova info atualiza a tabela
* $\gamma$: fator de desconto - quanto valorizo recompensas futuras
* $r$: recompensa imediada observada (d+1)
* $\max_{a'}Q(s',a')$: melhor valor esperado a partir do próximo estado

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Primeiro gráfico: Mapa de calor da Q-Table. 
# O que o agente efetivamente "decorou".
sns.heatmap(Q, annot=True, fmt=".3f", cmap="RdYlGn", center=0,
            xticklabels=["flat","long","short"],
            yticklabels=[f"s{i}" for i in range(n_states)], ax=axes[0])
axes[0].set_title("Q-table aprendida")

# Segundo gráfico: Convergência da recompensa.
# Exibe a pontuação total por episódio. Uma curva ascendente significa que ele está aprendendo.
# Aplicamos uma média móvel de 20 episódios (rolling) para suavizar a curva de aprendizado (tirar ruídos de exploração).
axes[1].plot(pd.Series(returns_per_episode).rolling(20).mean(), color="#0A2540")
axes[1].set_title("Recompensa acumulada / episodio (MM20)")
axes[1].set_xlabel("Episodio")
axes[1].set_ylabel("Soma de recompensas")
plt.tight_layout()
plt.show()

4. Avaliação Out-of-Sample
¶

In [ ]:
test = test.copy()

# Em modo "produção/teste", o agente não explora (epsilon = 0). Ele toma a decisão pura da tabela aprendida.
test["action"] = test["state"].map(lambda s: int(np.argmax(Q[int(s)]))) 

# Simulando os lucros baseado na ação escolhida
test["strategy_ret"] = test.apply(lambda r: reward(int(r["action"]), float(r["ret"])), axis=1)

# Cumprod (Produto Cumulativo): Transforma os retornos percentuais na evolução financeira do capital (Equity Curve).
test["equity_rl"] = (1 + test["strategy_ret"]).cumprod()
test["equity_bh"] = (1 + test["ret"]).cumprod() # Como teria sido se apenas comprasse o ativo e não fizesse nada (Buy & Hold).

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(test["date"], test["equity_rl"], label="Q-Learning", color="#0A2540", linewidth=2)
ax.plot(test["date"], test["equity_bh"], label="Buy & Hold", color="#1FB8CD", linewidth=2)
ax.set_title("Equity curve - Out-of-Sample")
ax.set_ylabel("Equity (1 = base)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cálculo do Índice de Sharpe Anualizado (mede a relação risco-retorno)
def sharpe(returns, periods=252):
    mu = returns.mean() * periods
    sd = returns.std() * (periods ** 0.5)
    return mu / sd if sd > 0 else 0.0

# Max Drawdown indica a maior perda consecutiva do topo ao fundo
def max_drawdown(equity):
    peak = equity.cummax()            # Vai salvando os topos históricos
    dd = (equity / peak) - 1          # Subtração simples para ver a queda do último topo
    return dd.min()                   # Retorna a maior queda (o valor mais negativo)

metrics = pd.DataFrame({
    "Q-Learning": [sharpe(test["strategy_ret"]), max_drawdown(test["equity_rl"]), test["equity_rl"].iloc[-1] - 1],
    "Buy & Hold": [sharpe(test["ret"]),          max_drawdown(test["equity_bh"]), test["equity_bh"].iloc[-1] - 1],
}, index=["Sharpe (anual)", "Max Drawdown", "Retorno total"])

# Imprime o DataFrame para comparativo entre o robô RL e a estratégia estática
print(metrics.round(3))

* sharpe anualizado = (retorno medio anualizado)/(volatilidade anualizada)
* Max Drawndown = pior queda acumulada do pico ao vale. 

5. Análise de Sensibilidade
¶

In [ ]:
# Vamos testar de forma exaustiva quais seriam os melhores valores para Alpha e Gamma
alphas = [0.05, 0.10, 0.20]
gammas = [0.80, 0.90, 0.99]
grid   = np.zeros((len(alphas), len(gammas)))

for ia, a_lr in enumerate(alphas):
    for ig, g_df in enumerate(gammas):
        Q2 = np.zeros((n_states, n_actions)) # Uma Q-Table em branco para cada combinação
        
        # Treinamento reduzido (150 episódios e Epsilon fixo em 10% por questões de agilidade no GridSearch)
        for ep in range(150):
            eps = 0.1
            for i in range(len(train) - 1):
                s   = int(train["state"].iat[i])
                s_n = int(train["state"].iat[i+1])
                r_n = float(train["ret"].iat[i+1])
                
                # Epsilon-Greedy
                if np.random.rand() < eps:
                    a = np.random.randint(n_actions)
                else:
                    a = int(np.argmax(Q2[s]))
                r = reward(a, r_n)
                
                # Atualização Q-Table usando a_lr e g_df dinâmicos do loop
                Q2[s, a] += a_lr * (r + g_df * Q2[s_n].max() - Q2[s, a])
        
        # Coleta das métricas (no dataset de teste) para o modelo da vez
        acts = test["state"].map(lambda s: int(np.argmax(Q2[int(s)])))
        rets = [reward(int(a), float(r)) for a, r in zip(acts, test["ret"])]
        
        # Salvamos o Índice de Sharpe no grid correspondente
        grid[ia, ig] = sharpe(pd.Series(rets))

# Plotando os resultados do Grid Search para definir os melhores parâmetros visualmente
fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(grid, annot=True, fmt=".2f", cmap="RdYlGn", center=0,
            xticklabels=gammas, yticklabels=alphas, ax=ax)
ax.set_xlabel("gamma")
ax.set_ylabel("alpha")
ax.set_title("Sharpe out-of-sample por (alpha, gamma)")
plt.tight_layout()

Em dados reais:

* $\alpha$ muito alto: instável
* $\alpha$ muito baixo: nao converge
* $\gamma$ alto: boa visao de longo prazo
* $\gamma$ baixo: míope

6. Exercícios propostos


Reescreva a funcao reward e subtraia 0.001 do reward em mudança de ação. Faca as alteracoes necessárias no treinamento


In [ ]:
# resposta